# Scenario: Uncovering the "Overconfident" AI

In [2]:
import pandas as pd
import sqlite3
# Creating the dataset tracking AI confidence scores against the human doctor's ground truth
diagnostic_data = {
    "imaging_id": [4001, 4002, 4003, 4004, 4005, 4006],
    "patient_id": ["P-31", "P-32", "P-33", "P-34", "P-35", "P-36"],
    "ai_confidence_score": [0.92, 0.45, 0.89, 0.94, 0.30, 0.87], # Probability from 0.0 to 1.0
    "ai_prediction": ["Pneumonia", "Normal", "Pneumonia", "Pneumonia", "Normal", "Pneumonia"],
    "doctor_final_diagnosis": ["Pneumonia", "Normal", "Normal", "Pneumonia", "Normal", "Normal"] # Look at 4003 and 4006!
}
# adding the dataset to DataFrame
df_diagnostic_data = pd.DataFrame(diagnostic_data)
# creating sql and save dataframe in temp memory
connt = sqlite3.connect(":memory:")
df_diagnostic_data.to_sql("imaging_logs", connt, index = False, if_exists = "replace")
# funtion to run the query
def run_query(query):
    return pd.read_sql_query(query, connt)
print("**************************** Confidence Threshold Audit Database is ready! **********")

**************************** Confidence Threshold Audit Database is ready! **********


# solating Overconfident Errors

In [5]:
# query for all data to review
all_data = "SELECT * FROM imaging_logs"
print("******************************* all data to review **************")
display(run_query(all_data))
print()
"""SQL query to extract all patient records where the AI's ai_confidence_score was greater than or equal to 0.85, 
but the ai_prediction did NOT match the doctor_final_diagnosis"""
ai_miss_match_prediction = """
SELECT 
    patient_id, 
    ai_confidence_score,
    ai_prediction,
    doctor_final_diagnosis
FROM imaging_logs
WHERE ai_confidence_score >= 0.85 AND ai_prediction != doctor_final_diagnosis
"""
print("********************************** ai_miss_match_prediction ******************")
display(run_query(ai_miss_match_prediction))

******************************* all data to review **************


,imaging_id,patient_id,ai_confidence_score,ai_prediction,doctor_final_diagnosis
0,4001,P-31,0.92,Pneumonia,Pneumonia
1,4002,P-32,0.45,Normal,Normal
2,4003,P-33,0.89,Pneumonia,Normal
3,4004,P-34,0.94,Pneumonia,Pneumonia
4,4005,P-35,0.30,Normal,Normal
5,4006,P-36,0.87,Pneumonia,Normal



********************************** ai_miss_match_prediction ******************


,patient_id,ai_confidence_score,ai_prediction,doctor_final_diagnosis
0,P-33,0.89,Pneumonia,Normal
1,P-36,0.87,Pneumonia,Normal


# Calculating the Average Confidence of Mistakes

In [6]:
# SQL query to calculate the average confidence score (AVG(ai_confidence_score)) specifically for the rows where the AI's prediction was incorrect.
avg_error_confidence = """
SELECT AVG(ai_confidence_score) AS avg_error_confidence
FROM imaging_logs
WHERE ai_prediction != doctor_final_diagnosis
"""
print("******************************* avg_error_confidence ***************")
display(run_query(avg_error_confidence))

******************************* avg_error_confidence ***************


,avg_error_confidence
0,0.88
